In [3]:
import sys
print(sys.version)


3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [4]:
!pip install tensorflow


   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 4.2/351.2 MB 21.4 MB/s eta 0:00:17
   - -------------------------------------- 10.2/351.2 MB 25.8 MB/s eta 0:00:14
   - -------------------------------------- 15.7/351.2 MB 25.5 MB/s eta 0:00:14
   -- ------------------------------------- 22.0/351.2 MB 26.3 MB/s eta 0:00:13
   --- ------------------------------------ 28.3/351.2 MB 26.6 MB/s eta 0:00:13
   --- ------------------------------------ 34.1/351.2 MB 27.0 MB/s eta 0:00:12
   ---- ----------------------------------- 40.6/351.2 MB 27.8 MB/s eta 0:00:12
   ----- ---------------------------------- 47.2/351.2 MB 28.2 MB/s eta 0:00:11
   ------ --------------------------------- 53.2/351.2 MB 28.5 MB/s eta 0:00:11
   ------ --------------------------------- 60.8/351.2 MB 29.2 MB/s eta 0:00:10
   ------- -------------------------------- 67.4/351.2 MB 29.4 MB/s eta 0:00:10
   -------- ------------------------------- 73.1/3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [8]:
import tensorflow as tf
print(tf.__version__)


2.21.0


In [9]:
import sys
print(sys.executable)


C:\Users\USER\anaconda3\python.exe


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import ResNet50
import numpy as np

# 1. Setup Data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 2. Define Model Architectures

def get_lenet():
    return models.Sequential([
        layers.Conv2D(6, (5, 5), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D(),
        layers.Conv2D(16, (5, 5), activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(120, activation='relu'),
        layers.Dense(84, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name="LeNet-5")

def get_alexnet():
    # Modified for 32x32 input
    return models.Sequential([
        layers.Conv2D(48, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(192, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(192, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1024, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name="AlexNet_Modified")

def get_vgg16():
    # Standard VGG16 blocks
    model = models.Sequential(name="VGG-16")
    cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M'] # Reduced depth for CIFAR
    model.add(layers.Input(shape=(32, 32, 3)))
    for v in cfg:
        if v == 'M':
            model.add(layers.MaxPooling2D((2, 2)))
        else:
            model.add(layers.Conv2D(v, (3, 3), padding='same', activation='relu'))
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dense(10, activation='softmax'))
    return model

def get_resnet50():
    # Using Keras built-in ResNet50
    base_resnet = ResNet50(include_top=False, weights=None, input_shape=(32, 32, 3))
    model = models.Sequential([
        base_resnet,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ], name="ResNet-50")
    return model

# 3. Training Loop
model_factories = [get_lenet, get_alexnet, get_vgg16, get_resnet50]
results = {}

for factory in model_factories:
    model = factory()
    print(f"\n--- Training {model.name} ---")
    
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    
    # Training for 5 epochs only for demonstration (Increase to 50+ for full accuracy)
    model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)
    
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results[model.name] = acc
    print(f"{model.name} Test Accuracy: {acc:.4f}")

# 4. Final Comparison
print("\n" + "="*30)
print("FINAL ACCURACY COMPARISON")
print("="*30)
for name, acc in results.items():
    print(f"{name:20}: {acc*100:.2f}%")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 16s 0us/step


C:\Users\USER\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- Training LeNet-5 ---
Epoch 1/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.3305 - loss: 1.8269 - val_accuracy: 0.4194 - val_loss: 1.5994
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4457 - loss: 1.5404 - val_accuracy: 0.4404 - val_loss: 1.5132
Epoch 3/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.4924 - loss: 1.4107 - val_accuracy: 0.4974 - val_loss: 1.3876
Epoch 4/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.5273 - loss: 1.3167 - val_accuracy: 0.5506 - val_loss: 1.2685
Epoch 5/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.5511 - loss: 1.2578 - val_accuracy: 0.5526 - val_loss: 1.2505
LeNet-5 Test Accuracy: 0.5490

--- Training AlexNet_Modified ---
Epoch 1/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 97s 267ms/step - accuracy: 0.3563 - loss: 1.7099 - val_accuracy: 0.5282 - val_loss: 1.2747
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 108s 307ms/step - accuracy: 0.5519 - loss: 1.2304 - val_accuracy: 0.6356 - val_loss: 1.0228
Epoch 3/5
